In [1]:
import pandas as pd
ts_data=pd.read_parquet("data/transformed/ts_data_2022_01.parquet")
ts_data

,pickup_hour,rides,pickup_location_id
0,2022-01-01 00:00:00,11,4
1,2022-01-01 01:00:00,15,4
2,2022-01-01 02:00:00,26,4
3,2022-01-01 03:00:00,8,4
4,2022-01-01 04:00:00,9,4
...,...,...,...
191203,2022-01-31 19:00:00,0,176
191204,2022-01-31 20:00:00,0,176
191205,2022-01-31 21:00:00,0,176
191206,2022-01-31 22:00:00,0,176


In [2]:
# filtering out for one location (central park)
ts_data_one_location = ts_data.loc[ts_data.pickup_location_id == 43, :].reset_index(drop=True)
ts_data_one_location

,pickup_hour,rides,pickup_location_id
0,2022-01-01 00:00:00,97,43
1,2022-01-01 01:00:00,60,43
2,2022-01-01 02:00:00,22,43
3,2022-01-01 03:00:00,8,43
4,2022-01-01 04:00:00,6,43
...,...,...,...
739,2022-01-31 19:00:00,61,43
740,2022-01-31 20:00:00,73,43
741,2022-01-31 21:00:00,33,43
742,2022-01-31 22:00:00,21,43


In [ ]:
# get_cutoff_indices function to be used for slicing step in sliding-window approach

def get_cutoff_indices(
        data: pd.DataFrame,
        n_features: int,
        step_size: int,

) -> list:

    stop_position = len(data) -1 

    # start the first sub-sequence at position 0
    subseq_first_idx = 0
    subseq_mid_idx = n_features
    subseq_last_idx = n_features + 1
    indices = []

    while subseq_last_idx <= stop_position:

        indices.append((subseq_first_idx, subseq_mid_idx, subseq_last_idx))

        subseq_first_idx += step_size
        subseq_mid_idx += step_size
        subseq_last_idx += step_size

    return indices

In [4]:
n_features = 24 # using the whole previous 24 hours as features
step_size = 1

indices = get_cutoff_indices(
    ts_data_one_location,
    n_features,
    step_size
)
indices

indices [:5]

[(0, 24, 25), (1, 25, 26), (2, 26, 27), (3, 27, 28), (4, 28, 29)]

In [5]:
# implementing the slicing step
import numpy as np
n_examples= len(indices)
x= np.ndarray(shape=(n_examples, n_features), dtype = np.float32)
y= np.ndarray(shape=(n_examples), dtype= np.float32)
pickup_hours= []

for i, idx in enumerate(indices):
    x[i,:]= ts_data_one_location.iloc[idx[0]:idx[1]]['rides'].values
    y[i]= ts_data_one_location.iloc[idx[1]]['rides']
    pickup_hours.append(ts_data_one_location.iloc[idx[1]]['pickup_hour'])


In [6]:
print(f'{x.shape=}')
print(f'{x=}')
print(f'{y.shape=}')
print(f'{y=}')
print(f'{pickup_hours=}')



x.shape=(719, 24)
x=array([[ 97.,  60.,  22., ...,  16.,  18.,   6.],
       [ 60.,  22.,   8., ...,  18.,   6.,   3.],
       [ 22.,   8.,   6., ...,   6.,   3.,   1.],
       ...,
       [ 28.,  16.,  13., ..., 102.,  66.,  61.],
       [ 16.,  13.,   8., ...,  66.,  61.,  73.],
       [ 13.,   8.,   1., ...,  61.,  73.,  33.]],
      shape=(719, 24), dtype=float32)
y.shape=(719,)
y=array([  3.,   1.,   1.,   0.,   0.,   3.,   4.,  11.,  19.,  31.,  44.,
        61.,  80., 114., 125., 122., 143.,  86.,  34.,  29.,  25.,  23.,
        25.,   7.,   1.,   0.,   0.,   1.,   0.,   3.,  12.,  29.,  62.,
        72.,  78.,  81.,  97., 112., 138., 149., 138., 117.,  50.,  36.,
        23.,  15.,  24.,   8.,   3.,   0.,   0.,   0.,   2.,   3.,   6.,
        43.,  90.,  89.,  98.,  95.,  84.,  97., 135., 129., 156., 132.,
        48.,  45.,  17.,  25.,  21.,  16.,   2.,   5.,   0.,   0.,   1.,
         6.,  10.,  44.,  93.,  76., 121., 132.,  81.,  83., 106., 122.,
       114.,  98.,  64.,  56

In [7]:
# transform x as numpy array into a pandas dataframe
features_one_location= pd.DataFrame(
    x, 
    columns=[f'rides_previous_{i+1}_hour' for i in reversed(range(n_features))]
)
features_one_location

,rides_previous_24_hour,rides_previous_23_hour,rides_previous_22_hour,rides_previous_21_hour,rides_previous_20_hour,rides_previous_19_hour,rides_previous_18_hour,rides_previous_17_hour,rides_previous_16_hour,rides_previous_15_hour,...,rides_previous_10_hour,rides_previous_9_hour,rides_previous_8_hour,rides_previous_7_hour,rides_previous_6_hour,rides_previous_5_hour,rides_previous_4_hour,rides_previous_3_hour,rides_previous_2_hour,rides_previous_1_hour
0,97.0,60.0,22.0,8.0,6.0,5.0,3.0,10.0,7.0,19.0,...,70.0,94.0,87.0,73.0,34.0,32.0,22.0,16.0,18.0,6.0
1,60.0,22.0,8.0,6.0,5.0,3.0,10.0,7.0,19.0,24.0,...,94.0,87.0,73.0,34.0,32.0,22.0,16.0,18.0,6.0,3.0
2,22.0,8.0,6.0,5.0,3.0,10.0,7.0,19.0,24.0,39.0,...,87.0,73.0,34.0,32.0,22.0,16.0,18.0,6.0,3.0,1.0
3,8.0,6.0,5.0,3.0,10.0,7.0,19.0,24.0,39.0,35.0,...,73.0,34.0,32.0,22.0,16.0,18.0,6.0,3.0,1.0,1.0
4,6.0,5.0,3.0,10.0,7.0,19.0,24.0,39.0,35.0,77.0,...,34.0,32.0,22.0,16.0,18.0,6.0,3.0,1.0,1.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
714,52.0,36.0,28.0,16.0,13.0,8.0,1.0,1.0,2.0,1.0,...,78.0,74.0,66.0,91.0,117.0,100.0,106.0,147.0,121.0,102.0
715,36.0,28.0,16.0,13.0,8.0,1.0,1.0,2.0,1.0,1.0,...,74.0,66.0,91.0,117.0,100.0,106.0,147.0,121.0,102.0,66.0
716,28.0,16.0,13.0,8.0,1.0,1.0,2.0,1.0,1.0,4.0,...,66.0,91.0,117.0,100.0,106.0,147.0,121.0,102.0,66.0,61.0
717,16.0,13.0,8.0,1.0,1.0,2.0,1.0,1.0,4.0,9.0,...,91.0,117.0,100.0,106.0,147.0,121.0,102.0,66.0,61.0,73.0


In [8]:
# transform y as numpy array into a pandas dataframe
targets_one_location= pd.DataFrame(y, columns=[f'target_rides_next_hour'])
targets_one_location

,target_rides_next_hour
0,3.0
1,1.0
2,1.0
3,0.0
4,0.0
...,...
714,66.0
715,61.0
716,73.0
717,33.0


In [9]:
''' creating a function to integrate all this data processing, converting the time series data 
into tabular data for all locations '''

from tqdm import tqdm

def transform_ts_data_into_features_and_target(
        ts_data: pd.DataFrame,
        input_seq_len: int,
        step_size: int
) -> pd.DataFrame:
    """
    slices and transposes a data from time-series format into a (features, target)
    format that we can use to train supervised ML models
    """

    assert set(ts_data.columns)== {'pickup_hour', 'rides', 'pickup_location_id'}

    location_ids= ts_data['pickup_location_id'].unique()
    features= pd.DataFrame()
    targets= pd.DataFrame()

    for location_id in tqdm(location_ids):

        # keep only ts data for this 'location_id'
        ts_data_one_location= ts_data.loc[
            ts_data.pickup_location_id== location_id, ['pickup_hour', 'rides']
        ]

        # pre-compute cutoff indices to split dataframe rows
        indices= get_cutoff_indices(
            ts_data_one_location,
            input_seq_len,
            step_size
        )

        # slice and transpose data into numpy arrays for features and targets
        n_examples= len(indices)
        n_examples= len(indices)
        x= np.ndarray(shape=(n_examples, input_seq_len), dtype = np.float32)
        y= np.ndarray(shape=(n_examples), dtype= np.float32)
        pickup_hours= []

        for i, idx in enumerate(indices):
            x[i,:]= ts_data_one_location.iloc[idx[0]:idx[1]]['rides'].values
            y[i]= ts_data_one_location.iloc[idx[1]]['rides']
            pickup_hours.append(ts_data_one_location.iloc[idx[1]]['pickup_hour'])

        # numpy -> pandas
        features_one_location= pd.DataFrame(
            x,
            columns=[f'rides_previous_{i+1}_hour' for i in reversed(range(input_seq_len))]
        )

        # numpy -> pandas
        targets_one_location= pd.DataFrame(y, columns=[f'target_rides_next_hour'])

        # concatenate results
        features= pd.concat([features, features_one_location])
        targets= pd.concat([targets, targets_one_location])

    features.reset_index(inplace= True, drop= True)
    targets.reset_index(inplace= True, drop= True)

    return features, targets['target_rides_next_hour']


In [10]:
features, targets= transform_ts_data_into_features_and_target(
    ts_data,
    input_seq_len= 24*7*1, # one week of rides history
    step_size= 24
)

print(f'{features.shape=}')
print(f'{targets.shape=}')

  0%|          | 0/257 [00:00<?, ?it/s]

100%|██████████| 257/257 [00:05<00:00, 44.40it/s]

features.shape=(6168, 168)
targets.shape=(6168,)
